# Taller aplicado: PCA vs PPCA en un campus inteligente

**Curso:** Estadística Multivariable Aplicada para Ciencia de Datos  
**Duración de trabajo:** 60 minutos  
**Sustentación:** 5 minutos por grupo  
**Modalidad:** trabajo en computador, en grupos de 2 estudiantes  

## Propósito

En este taller van a analizar datos de sensores de un edificio universitario. El objetivo es comparar **PCA** y **PPCA** como herramientas para descubrir una estructura latente detrás de varias variables observadas.

La idea central no es solamente “reducir dimensión”, sino proponer una interpretación del **mundo latente** que podría estar generando los datos.


## Contexto aplicado

La universidad está probando un sistema de monitoreo para un edificio de aulas. Cada registro corresponde a un bloque horario observado durante varios días. Para cada bloque se miden variables ambientales, de ocupación y consumo energético.

El equipo de analítica debe responder una pregunta:

> ¿Existen pocas dimensiones latentes que resuman el estado operativo y ambiental del edificio?

El archivo de datos asociado al taller es:

`datos_campus_ambiental_ppca.csv`

Variables disponibles:

| Variable | Descripción |
|---|---|
| `PM25_ug_m3` | concentración de material particulado fino |
| `CO2_ppm` | concentración de dióxido de carbono |
| `Ruido_dB` | nivel de ruido |
| `Temperatura_C` | temperatura del aula o zona |
| `Humedad_pct` | humedad relativa |
| `Ocupacion_personas` | número estimado de personas |
| `Energia_kWh` | consumo energético del bloque |
| `Ventilacion_indice` | indicador operativo de ventilación |

Las variables `dia` y `turno` son descriptivas y no deben usarse directamente para ajustar PCA o PPCA.


## Entregable al final de la hora

Cada grupo debe entregar este notebook diligenciado con:

1. Exploración inicial de los datos.
2. Aplicación de PCA.
3. Aplicación o formulación de PPCA.
4. Comparación interpretativa entre PCA y PPCA.
5. Propuesta del mundo latente.
6. Conclusión ejecutiva para la universidad.

La sustentación debe durar máximo **5 minutos** y debe responder:

> ¿Qué aprendimos del edificio y qué modelo explica mejor la estructura de los datos?


---
# Parte 1. Lectura y comprensión del problema

## Pregunta 1

Carguen el archivo de datos y describan brevemente qué representa una fila y qué representa una columna.

Escriban aquí su respuesta en lenguaje claro.


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Generación de datos sintéticos realistas
# Contexto: Campus inteligente con sensores ambientales
# ============================================================

np.random.seed(123)

# Número de registros
n = 1200

# Variables de contexto
zonas = [
    "Biblioteca",
    "Aula grande",
    "Laboratorio de cómputo",
    "Cafetería",
    "Zona administrativa",
    "Pasillo principal"
]

franjas = ["Mañana", "Mediodía", "Tarde", "Noche"]

zona = np.random.choice(zonas, size=n, p=[0.18, 0.22, 0.18, 0.16, 0.14, 0.12])
franja = np.random.choice(franjas, size=n, p=[0.30, 0.25, 0.30, 0.15])

# Día de la semana: 0=lunes, ..., 6=domingo
dia_semana = np.random.choice(range(7), size=n, p=[0.16, 0.17, 0.17, 0.17, 0.16, 0.10, 0.07])

# ============================================================
# Mundo latente simulado
# ============================================================

# Factor 1: ocupación / intensidad de uso del espacio
uso_base = np.random.normal(0, 1, n)

# Factor 2: condición ambiental externa
clima_base = np.random.normal(0, 1, n)

# Factor 3: ventilación / calidad de circulación del aire
ventilacion_base = np.random.normal(0, 1, n)

# Ajustes por zona
ajuste_zona_uso = {
    "Biblioteca": 0.4,
    "Aula grande": 0.8,
    "Laboratorio de cómputo": 0.6,
    "Cafetería": 1.0,
    "Zona administrativa": 0.2,
    "Pasillo principal": 0.7
}

ajuste_zona_ruido = {
    "Biblioteca": -0.8,
    "Aula grande": 0.3,
    "Laboratorio de cómputo": 0.1,
    "Cafetería": 1.0,
    "Zona administrativa": -0.2,
    "Pasillo principal": 0.6
}

ajuste_zona_ventilacion = {
    "Biblioteca": 0.2,
    "Aula grande": -0.1,
    "Laboratorio de cómputo": -0.3,
    "Cafetería": 0.1,
    "Zona administrativa": 0.3,
    "Pasillo principal": 0.6
}

# Ajustes por franja horaria
ajuste_franja_uso = {
    "Mañana": 0.5,
    "Mediodía": 0.9,
    "Tarde": 0.7,
    "Noche": -0.3
}

ajuste_franja_temperatura = {
    "Mañana": -0.4,
    "Mediodía": 0.8,
    "Tarde": 0.5,
    "Noche": -0.6
}

uso = (
    uso_base
    + np.array([ajuste_zona_uso[z] for z in zona])
    + np.array([ajuste_franja_uso[f] for f in franja])
    - 0.4 * (dia_semana >= 5)
)

clima = (
    clima_base
    + np.array([ajuste_franja_temperatura[f] for f in franja])
)

ventilacion = (
    ventilacion_base
    + np.array([ajuste_zona_ventilacion[z] for z in zona])
)

# ============================================================
# Variables observadas
# ============================================================

temperatura = 20 + 1.8 * clima + 0.7 * uso - 0.5 * ventilacion + np.random.normal(0, 0.7, n)

humedad = 58 - 2.2 * clima + 0.8 * ventilacion + np.random.normal(0, 2.5, n)

co2 = 500 + 120 * uso - 70 * ventilacion + np.random.normal(0, 45, n)

ruido_db = (
    45
    + 6.5 * uso
    + np.array([4 * ajuste_zona_ruido[z] for z in zona])
    + np.random.normal(0, 3.5, n)
)

luminosidad = (
    420
    + 60 * clima
    + 35 * (franja == "Mañana")
    + 25 * (franja == "Tarde")
    - 80 * (franja == "Noche")
    + np.random.normal(0, 45, n)
)

ocupacion_estimada = (
    30
    + 14 * uso
    + np.random.normal(0, 5, n)
)

consumo_energia = (
    18
    + 1.8 * temperatura
    + 0.06 * ocupacion_estimada
    + 0.012 * luminosidad
    + np.random.normal(0, 2.5, n)
)

material_particulado = (
    12
    + 2.5 * uso
    - 1.2 * ventilacion
    + 0.8 * (zona == "Cafetería")
    + np.random.normal(0, 2, n)
)

indice_confort = (
    80
    - 0.9 * np.abs(temperatura - 22)
    - 0.035 * co2
    - 0.25 * ruido_db
    + 0.04 * luminosidad
    + np.random.normal(0, 3, n)
)

# ============================================================
# Corrección de rangos realistas
# ============================================================

temperatura = np.clip(temperatura, 16, 30)
humedad = np.clip(humedad, 35, 85)
co2 = np.clip(co2, 350, 1400)
ruido_db = np.clip(ruido_db, 30, 85)
luminosidad = np.clip(luminosidad, 80, 850)
ocupacion_estimada = np.clip(ocupacion_estimada, 0, 120)
consumo_energia = np.clip(consumo_energia, 20, 95)
material_particulado = np.clip(material_particulado, 3, 45)
indice_confort = np.clip(indice_confort, 0, 100)

# ============================================================
# Construcción del DataFrame
# ============================================================

datos = pd.DataFrame({
    "zona": zona,
    "franja": franja,
    "dia_semana": dia_semana,
    "temperatura": temperatura,
    "humedad": humedad,
    "co2": co2,
    "ruido_db": ruido_db,
    "luminosidad": luminosidad,
    "ocupacion_estimada": ocupacion_estimada,
    "consumo_energia": consumo_energia,
    "material_particulado": material_particulado,
    "indice_confort": indice_confort
})

# Redondeo
columnas_numericas = [
    "temperatura",
    "humedad",
    "co2",
    "ruido_db",
    "luminosidad",
    "ocupacion_estimada",
    "consumo_energia",
    "material_particulado",
    "indice_confort"
]

datos[columnas_numericas] = datos[columnas_numericas].round(2)

# Mostrar primeras filas
datos.head()

,zona,franja,dia_semana,temperatura,humedad,co2,ruido_db,luminosidad,ocupacion_estimada,consumo_energia,material_particulado,indice_confort
0,Cafetería,Tarde,3,21.61,60.93,592.53,52.93,468.91,33.55,66.40,16.12,69.45
1,Aula grande,Mediodía,6,23.59,53.66,546.72,38.18,452.71,27.63,69.33,8.35,65.95
2,Aula grande,Noche,1,19.88,63.61,350.00,43.54,316.86,18.28,62.65,10.82,71.37
3,Laboratorio de cómputo,Tarde,0,24.33,54.95,711.19,62.01,535.54,51.80,74.96,18.18,64.04
4,Cafetería,Tarde,1,21.07,55.66,592.76,54.58,414.62,61.27,62.37,13.78,60.31


## Pregunta 2

Identifiquen qué variables son cuantitativas y cuáles son variables de identificación o contexto. Expliquen cuáles usarán para PCA y PPCA, y cuáles no.

**Respuesta:**


## Pregunta 3

Antes de hacer PCA o PPCA, revisen si hay valores faltantes.

Respondan:

- ¿En qué variables aparecen valores faltantes?
- ¿Por qué esto puede ser un problema para PCA clásico?
- ¿Por qué PPCA podría ser conceptualmente atractivo cuando hay ruido o datos incompletos?

**Respuesta:**


---
# Parte 2. Exploración multivariada

## Pregunta 4

Construyan una exploración gráfica o descriptiva de las variables cuantitativas.

Deben incluir al menos:

- una tabla de resumen descriptivo;
- una visualización que permita ver relaciones entre variables;
- una matriz de correlaciones o una representación equivalente.

Luego escriban una interpretación breve.

**Interpretación:**


## Pregunta 5

A partir de la exploración inicial, propongan **dos o tres posibles factores latentes** que podrían explicar las relaciones entre las variables.

Por ejemplo, no basta decir “componente 1” o “componente 2”. Deben proponer nombres interpretables, como:

- presión de ocupación;
- carga ambiental;
- confort térmico;
- eficiencia operativa;
- otro nombre defendible.

**Mundo latente propuesto antes de modelar:**


---
# Parte 3. PCA clásico

## Pregunta 6

Estandaricen las variables cuantitativas y ajusten un PCA.

Justifiquen por qué la estandarización es necesaria en este problema.

**Justificación:**


## Pregunta 7

Analicen la varianza explicada por las componentes principales.

Respondan:

- ¿Cuántas componentes conservarían?
- ¿Qué criterio usaron?
- ¿La reducción de dimensión parece razonable?

Pueden apoyarse en una tabla, un gráfico de varianza explicada o ambos.

**Respuesta:**


## Pregunta 8

Interpreten las cargas de las primeras componentes principales.

Para cada componente retenida, indiquen:

- variables con mayor peso positivo;
- variables con mayor peso negativo;
- nombre interpretativo de la componente;
- sentido aplicado de la componente.

**Interpretación:**


## Pregunta 9

Construyan una visualización de los datos proyectados sobre las dos primeras componentes principales.

Si lo consideran útil, coloreen los puntos por `turno`.

Respondan:

- ¿Aparecen patrones por turno?
- ¿Hay observaciones atípicas?
- ¿Qué podría significar una observación extrema en este contexto?

**Respuesta:**


---
# Parte 4. PPCA como modelo generativo

## Recordatorio conceptual

En PPCA se supone que cada observación $x_i$ se genera desde una variable latente $z_i$ mediante el modelo:

$$
x_i = Wz_i + \mu +
\varepsilon_i,
$$

con

$$
z_i \sim N(0,I_k), \qquad
\varepsilon_i \sim N(0,\sigma^2 I_p).
$$

Por tanto,

$$
x_i \sim N(\mu, WW^T + \sigma^2 I_p).
$$

La matriz $W$ conecta el mundo latente con las variables observadas, mientras que $\sigma^2$ representa ruido isotrópico no explicado por los factores latentes.


## Pregunta 10

Formulen PPCA para este problema.

Definan claramente:

- qué representa $x_i$;
- qué representa $z_i$;
- qué dimensión latente $k$ proponen;
- qué significado aplicado tendría cada coordenada de $z_i$;
- qué representa el ruido $
\varepsilon_i$ en el contexto de sensores ambientales.

**Formulación:**


## Pregunta 11

Comparen la lógica de PCA y PPCA.

Completen la siguiente tabla en sus propias palabras:

| Aspecto | PCA | PPCA |
|---|---|---|
| Naturaleza del método |  |  |
| Tratamiento del ruido |  |  |
| Interpretación de componentes o factores |  |  |
| Manejo conceptual de incertidumbre |  |  |
| Utilidad para datos incompletos o ruidosos |  |  |
| Ventaja principal en este caso |  |  |
| Limitación principal en este caso |  |  |


## Pregunta 12

Ajusten o aproximen un modelo PPCA con la misma dimensión latente seleccionada para PCA.

Luego comparen:

- estructura latente obtenida;
- reconstrucción de los datos;
- interpretación de factores;
- sensibilidad frente a ruido o valores faltantes.

No se espera una implementación perfecta desde cero. Lo importante es que la comparación sea estadísticamente defendible.

**Resultados y comparación:**


---
# Parte 5. Comparación aplicada PCA vs PPCA

## Pregunta 13

Construyan una comparación final entre PCA y PPCA para este caso.

Respondan de forma concreta:

1. ¿Ambos modelos sugieren el mismo número de dimensiones latentes?
2. ¿Los factores encontrados tienen una interpretación similar?
3. ¿Cuál modelo comunica mejor el problema a un directivo no técnico?
4. ¿Cuál modelo es más coherente si los sensores tienen error de medición?
5. ¿Cuál usarían para monitoreo operativo del edificio?

**Respuesta:**


## Pregunta 14

Propongan una historia latente del edificio.

Deben escribir un párrafo en el que expliquen, como equipo de analítica, cuál podría ser el mecanismo oculto que genera los datos observados.

Ejemplo de estructura esperada:

> Los datos sugieren que el edificio puede entenderse mediante ... dimensiones latentes. La primera parece asociada con ..., porque .... La segunda parece asociada con ..., porque .... Esto permite interpretar que ....

**Historia latente propuesta:**


## Pregunta 15

Construyan una recomendación para la universidad.

Debe ser una recomendación aplicada, no puramente estadística, pero basada en los datos. Por ejemplo:

- ajustar ventilación en ciertos turnos;
- monitorear aulas con alta ocupación;
- diseñar alertas tempranas;
- combinar sensores para construir un índice de estado ambiental;
- revisar eficiencia energética.

**Recomendación:**


---
# Parte 6. Preparación de sustentación de 5 minutos

Cada grupo debe preparar una sustentación breve con esta estructura:

1. **Problema:** qué se quiso entender del edificio.
2. **Datos:** qué variables se analizaron.
3. **PCA:** qué componentes aparecieron y cómo se interpretan.
4. **PPCA:** qué aporta frente a PCA.
5. **Mundo latente:** cuál es la historia oculta propuesta.
6. **Decisión:** qué recomendarían hacer.

## Diapositiva única sugerida

Pueden usar una sola diapositiva con cuatro bloques:

| Bloque | Contenido |
|---|---|
| 1 | Pregunta aplicada |
| 2 | Resultado PCA |
| 3 | Resultado PPCA |
| 4 | Recomendación final |


---
# Criterios de evaluación

| Criterio | Peso |
|---|---:|
| Comprensión del contexto aplicado | 20% |
| Exploración y preparación de datos | 20% |
| Interpretación de PCA | 20% |
| Comparación PCA vs PPCA | 25% |
| Claridad de la sustentación | 15% |

## Nota final

No se evaluará solamente que el código funcione. Se evaluará principalmente la capacidad de conectar el modelo estadístico con una interpretación aplicada y defendible.
